In [ ]:
# =============================================================================
# Part 5 : 공동 구매 패턴 분석 (Co-purchase Pattern Mining)
# =============================================================================
# 차별화 포인트: 상품을 독립적으로 예측하는 기존 방식에서 벗어나
# 실제 주문에서 함께 담기는 상품 묶음 패턴을 연관 규칙으로 발견
# → 장바구니 추천 / 번들 프로모션 / 재구매 모델 피처 보강에 활용

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
import gc
from itertools import combinations

warnings.filterwarnings('ignore')

def set_korean_font():
    for font in ['NanumGothic', 'Malgun Gothic', 'AppleGothic']:
        if font in {f.name for f in fm.fontManager.ttflist}:
            plt.rcParams['font.family'] = font
            break
    plt.rcParams['axes.unicode_minus'] = False

set_korean_font()

DATA_PATH = r'C:\Users\LG\K-Pick\data\prep\k-pick_total_v3.csv'

print("=" * 60)
print("Part 5 : 공동 구매 패턴 분석 (Co-purchase Pattern Mining)")
print("=" * 60)
print("차별화: 개별 상품 예측 → 함께 구매되는 묶음 패턴 발견")
print("  예) 우유 구매 시 시리얼도 높은 확률로 구매 → 세트 추천")

# =============================================================================
# 1. 바구니 데이터 로드 (order_id + product_id + label)
# =============================================================================
print("\n1. 바구니 데이터 로드")

basket_df = pd.read_csv(
    DATA_PATH,
    usecols=['order_id', 'product_id', 'label'],
    dtype={'order_id': 'int32', 'product_id': 'int32', 'label': 'int8'},
    low_memory=True,
)
print(f"전체 데이터 shape: {basket_df.shape}")

# 실제 구매 확정된 상품만 사용 (label=1)
purchased = basket_df[basket_df['label'] == 1][['order_id', 'product_id']].copy()
del basket_df; gc.collect()
print(f"실제 구매 행수: {purchased.shape[0]:,}")

# =============================================================================
# 2. 주문별 바구니 생성
# =============================================================================
print("\n2. 주문별 바구니 생성")

baskets = purchased.groupby('order_id')['product_id'].apply(list)
total_orders = len(baskets)
baskets = baskets[baskets.apply(len) >= 2]   # 2개 이상 상품 주문만 유효

print(f"전체 주문 수    : {total_orders:,}")
print(f"복수 상품 주문  : {len(baskets):,}  ({len(baskets)/total_orders*100:.1f}%)")
print(f"주문당 평균 상품: {baskets.apply(len).mean():.2f}개")

# 속도 최적화: 최대 100,000 주문 샘플링
MAX_ORDERS = 100_000
if len(baskets) > MAX_ORDERS:
    baskets = baskets.sample(MAX_ORDERS, random_state=42)
    print(f"속도 최적화: {MAX_ORDERS:,}건 샘플링")

# =============================================================================
# 3. 공동 구매 빈도 & 연관 규칙 지표 계산
# =============================================================================
print("\n3. 공동 구매 빈도 계산 중...")

co_purchase = {}   # (product_A, product_B) → 공동 구매 횟수
item_count  = {}   # product_id → 단독 구매 횟수

for products in baskets:
    products = list(set(products))   # 동일 주문 내 중복 제거
    for pid in products:
        item_count[pid] = item_count.get(pid, 0) + 1
    for a, b in combinations(sorted(products), 2):
        co_purchase[(a, b)] = co_purchase.get((a, b), 0) + 1

n_baskets = len(baskets)
print(f"고유 상품 수  : {len(item_count):,}")
print(f"공동 구매 쌍  : {len(co_purchase):,}")

# 연관 규칙 지표 계산
print("연관 규칙 지표 계산 중...")
rules = []
for (a, b), count in co_purchase.items():
    sup_ab = count / n_baskets
    sup_a  = item_count[a] / n_baskets
    sup_b  = item_count[b] / n_baskets
    rules.append({
        'product_A'    : a,
        'product_B'    : b,
        'co_count'     : count,
        'support'      : round(sup_ab, 6),
        'confidence_AB': round(count / item_count[a], 4),   # P(B|A): A 살 때 B도 살 확률
        'confidence_BA': round(count / item_count[b], 4),   # P(A|B)
        'lift'         : round(sup_ab / (sup_a * sup_b), 4) if sup_a * sup_b > 0 else 0,
    })

rules_df = pd.DataFrame(rules)
del co_purchase, item_count; gc.collect()

# =============================================================================
# 4. 상위 공동 구매 패턴 출력
# =============================================================================
print("\n" + "=" * 60)
print("Support 상위 20 공동 구매 쌍  (자주 함께 담기는 조합)")
print("=" * 60)
top_sup = rules_df.nlargest(20, 'support')
print(top_sup[['product_A', 'product_B', 'co_count', 'support', 'confidence_AB', 'lift']].to_string(index=False))

# Lift 기준: support ≥ 0.001 필터링 후 상위 20 (희귀하지만 강한 연관)
print("\n" + "=" * 60)
print("Lift 상위 20 공동 구매 쌍  (독립 대비 함께 구매될 배율, support ≥ 0.001 필터)")
print("=" * 60)
top_lift = rules_df[rules_df['support'] >= 0.001].nlargest(20, 'lift')
if len(top_lift) == 0:
    top_lift = rules_df.nlargest(20, 'lift')
print(top_lift[['product_A', 'product_B', 'co_count', 'support', 'confidence_AB', 'lift']].to_string(index=False))

# =============================================================================
# 5. 시각화
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 5-A. Support vs Lift 산점도
plot_df = rules_df[rules_df['support'] >= rules_df['support'].quantile(0.95)].copy()
sc = axes[0].scatter(
    plot_df['support'], plot_df['lift'],
    c=plot_df['confidence_AB'], cmap='YlOrRd', alpha=0.5, s=15
)
axes[0].axhline(1.0, color='gray', ls='--', lw=0.8, label='Lift=1 (독립)')
axes[0].set(xlabel='Support', ylabel='Lift',
            title='공동 구매 패턴: Support vs Lift\n(색상=Confidence A→B)')
axes[0].legend(fontsize=8)
plt.colorbar(sc, ax=axes[0], label='Confidence A→B')

# 5-B. Support 상위 15 쌍 막대그래프
top15 = rules_df.nlargest(15, 'support').copy()
pair_labels = [f"P{int(r.product_A)}+\nP{int(r.product_B)}" for _, r in top15.iterrows()]
axes[1].barh(pair_labels, top15['support'], color='#1D9E75', alpha=0.85)
axes[1].set(xlabel='Support', title='Support 상위 15\n공동 구매 쌍')

# 5-C. 공동 구매 히트맵 (상위 상품)
top_items = list(
    pd.concat([rules_df.nlargest(60, 'support')['product_A'],
               rules_df.nlargest(60, 'support')['product_B']])
    .value_counts().head(12).index
)
heat_data = pd.DataFrame(0.0, index=top_items, columns=top_items)
for _, row in rules_df.iterrows():
    if row['product_A'] in top_items and row['product_B'] in top_items:
        heat_data.loc[row['product_A'], row['product_B']] = row['support']
        heat_data.loc[row['product_B'], row['product_A']] = row['support']

im = axes[2].imshow(heat_data.values, cmap='Blues', aspect='auto')
tick_labels = [f'P{int(i)}' for i in top_items]
axes[2].set_xticks(range(len(top_items))); axes[2].set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=7)
axes[2].set_yticks(range(len(top_items))); axes[2].set_yticklabels(tick_labels, fontsize=7)
plt.colorbar(im, ax=axes[2], label='Support')
axes[2].set_title('상위 12 상품 공동 구매\n히트맵')

plt.tight_layout()
plt.show()

# =============================================================================
# 6. 공동 구매 피처 생성 (재구매 모델 보강용)
# =============================================================================
print("\n6. 공동 구매 피처 생성 — 재구매 모델 피처 엔지니어링 보강")

# 상품별 평균 Lift (다른 상품과 얼마나 자주 묶여 구매되는지)
prod_lift_mean = pd.concat([
    rules_df[['product_A', 'lift']].rename(columns={'product_A': 'product_id'}),
    rules_df[['product_B', 'lift']].rename(columns={'product_B': 'product_id'}),
]).groupby('product_id')['lift'].mean().rename('prod_mean_lift')

# 상품별 최고 Confidence (단일 방향 기준)
prod_conf_max = pd.concat([
    rules_df[['product_A', 'confidence_AB']].rename(
        columns={'product_A': 'product_id', 'confidence_AB': 'conf'}),
    rules_df[['product_B', 'confidence_BA']].rename(
        columns={'product_B': 'product_id', 'confidence_BA': 'conf'}),
]).groupby('product_id')['conf'].max().rename('prod_max_confidence')

copurchase_feats = pd.concat([prod_lift_mean, prod_conf_max], axis=1).reset_index()
print(f"공동 구매 피처 생성 완료: {copurchase_feats.shape}")
print(copurchase_feats.describe().round(4))
print("\n→ 이 피처를 k-pick_total_v3.csv에 product_id 기준 merge 하면")
print("  Part 1 피처 엔지니어링에서 재구매 모델 성능을 추가로 개선할 수 있습니다.")

# =============================================================================
# 7. 비즈니스 인사이트
# =============================================================================
print("\n" + "=" * 60)
print("비즈니스 인사이트: 공동 구매 패턴 활용 전략")
print("=" * 60)

if len(top_lift) > 0:
    best = top_lift.iloc[0]
    print(f"\n  최고 Lift 공동 구매 쌍:")
    print(f"    상품 {int(best.product_A)}  +  상품 {int(best.product_B)}")
    print(f"    Lift {best.lift:.2f}x  → 독립 구매 대비 {best.lift:.1f}배 자주 함께 구매")
    print(f"    A→B Confidence: {best.confidence_AB*100:.1f}%")

print("""
  활용 방안:
    1) 장바구니 추천  : 상품 A 담을 시 → "이 상품과 자주 함께 구매" 상품 B 노출
    2) 번들 프로모션  : Lift 상위 쌍을 묶음 할인 패키지로 구성
    3) 재고 연동      : 공동 구매 쌍 중 한 상품 품절 시 연관 상품 재고 사전 확보
    4) 모델 피처 보강 : prod_mean_lift / prod_max_confidence를 재구매 예측 피처로 추가
    5) 마케팅 타겟팅  : Part 4 구매 시점 예측 + 공동 구매 패턴 결합
                       → "곧 재구매할 고객"에게 연관 상품 세트 추천
""")
print("Part 5 완료 ✓  →  rules_df, copurchase_feats 유지")
